# 07 – Quarantine Report

Isolates rows that failed DQ checks into quarantine tables:
- Reads `gold.dq_report` for failed check metadata
- Re-evaluates the failed rules against silver data
- Stores offending rows with quarantine reason and timestamp

**Source:** `gold.dq_report`, `silver.{books, users, ratings}`
**Target:** `quarantine.{books_quarantine, users_quarantine, ratings_quarantine}`
**Config:** `config/07_quarantine_report.json`

### Gold Cleanup
After quarantine, removes flagged rows from gold tables via anti-join and recomputes summary aggregations to keep the trusted layer clean.

In [0]:
import json, sys, importlib

SETUP_CONFIG = "/Workspace/Users/tharcisio.leone@x1f-fink.one/poc/config/00_setup.json"
with open(SETUP_CONFIG) as f:
    setup = json.load(f)

sys.path.insert(0, setup["project_path"])

for mod in [m for m in list(sys.modules.keys()) if m.startswith("source")]:
    del sys.modules[mod]

QuarantineReport = importlib.import_module("source.07_quarantine_report").QuarantineReport

CONFIG_PATH = f"{setup['project_path']}/config/07_quarantine_report.json"

In [0]:
pipeline = QuarantineReport(config_path=CONFIG_PATH)
quarantine_dfs = pipeline.run()

In [0]:
for table_key, df in quarantine_dfs.items():
    count = df.count()
    print(f"quarantine.{table_key}_quarantine: {count} rows")
    if count > 0:
        for row in df.select("_quarantine_reason").distinct().collect():
            print(f"Reason: {row['_quarantine_reason']}")
        display(df.limit(10))

In [0]:
pipeline.clean_gold()
pipeline.verify_gold_cleanup()